# Quanta vida útil a troca por tempo joga fora

**Pergunta:** quando a vida útil varia muito de um equipamento para outro, quanto se
desperdiça trocando por tempo fixo — e quanto dá para aproveitar prevendo o fim de
cada um, sem aumentar o risco de falha?

Dados: *C-MAPSS Turbofan Engine Degradation Simulation*, NASA Prognostics Center of
Excellence — subconjunto FD001.

Este projeto é o par do D1. Lá, as falhas por desgaste aconteciam numa janela
estreita de tempo de uso, e a troca preventiva por tempo resolvia. Aqui a pergunta
é o caso oposto.

## 1. Carregar e descrever os dados

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold

SEMENTE = 42
COLUNAS = ["motor", "ciclo", "op1", "op2", "op3"] + [f"s{i}" for i in range(1, 22)]
treino = pd.read_csv("dados/train_FD001.txt", sep=r"\s+", header=None, names=COLUNAS)
teste = pd.read_csv("dados/test_FD001.txt", sep=r"\s+", header=None, names=COLUNAS)
rul_teste = pd.read_csv("dados/RUL_FD001.txt", header=None, names=["rul_real"])
rul_teste["motor"] = range(1, len(rul_teste) + 1)

print(f"Treino: {treino.motor.nunique()} motores, {len(treino):,} ciclos registrados, rodados até a falha")
print(f"Teste:  {teste.motor.nunique()} motores, interrompidos antes da falha, com a vida restante real informada")
print(f"Faltantes: {treino.isna().sum().sum()}")

Treino: 100 motores, 20,631 ciclos registrados, rodados até a falha
Teste:  100 motores, interrompidos antes da falha, com a vida restante real informada
Faltantes: 0


Cada linha é um ciclo de operação de um motor: 3 ajustes de operação e 21 sensores
(temperaturas, pressões, rotações). No treino, cada motor roda até falhar — então se
sabe a vida inteira de cada um.

In [2]:
constantes = [col for col in COLUNAS[2:] if treino[col].std() < 1e-3]
SENSORES = [col for col in COLUNAS[5:] if col not in constantes]
print("Sem variação (descartados):", ", ".join(constantes))
print(f"Sensores usados: {len(SENSORES)}")

Sem variação (descartados): op2, op3, s1, s5, s10, s16, s18, s19
Sensores usados: 15


## 2. A vida útil varia quanto?

In [3]:
vida = treino.groupby("motor")["ciclo"].max()
print(f"Vida útil dos 100 motores: de {vida.min()} a {vida.max()} ciclos (mediana {vida.median():.0f}, média {vida.mean():.1f})")
print(f"O motor que mais durou durou {vida.max() / vida.min():.1f} vezes o que menos durou.")
print()
print("No D1, as falhas por desgaste da ferramenta iam de 198 a 253 minutos:")
print(f"o maior valor era {253 / 198:.2f} vezes o menor.")

Vida útil dos 100 motores: de 128 a 362 ciclos (mediana 199, média 206.3)
O motor que mais durou durou 2.8 vezes o que menos durou.

No D1, as falhas por desgaste da ferramenta iam de 198 a 253 minutos:
o maior valor era 1.28 vezes o menor.


Essa é a diferença que decide tudo. Com janela estreita, uma data fixa de troca
pega quase todos perto do fim. Com janela larga, a data fixa precisa ser cedo o
bastante para o motor mais curto — e todos os outros saem com vida sobrando.

## 3. O modelo: prever quantos ciclos faltam

Para cada ciclo, o modelo estima a **vida restante** a partir dos sensores e das
médias móveis dos últimos 10 e 30 ciclos (a tendência pesa mais que a leitura do
instante).

A vida restante é limitada a 125 ciclos no treino — prática comum neste dataset:
no começo da vida o motor está saudável e não há sinal de desgaste para distinguir
"faltam 300" de "faltam 200". O que importa para a decisão é a reta final.

In [4]:
def com_tendencia(d):
    d = d.sort_values(["motor", "ciclo"]).copy()
    g = d.groupby("motor")
    for s in SENSORES:
        d[f"{s}_m10"] = g[s].transform(lambda x: x.rolling(10, min_periods=1).mean())
        d[f"{s}_m30"] = g[s].transform(lambda x: x.rolling(30, min_periods=1).mean())
    return d

treino = com_tendencia(treino)
teste = com_tendencia(teste)
ENTRADAS = SENSORES + [f"{s}_m10" for s in SENSORES] + [f"{s}_m30" for s in SENSORES] + ["ciclo"]
TETO = 125
treino["vida_restante"] = treino.motor.map(vida) - treino.ciclo
treino["alvo"] = treino.vida_restante.clip(upper=TETO)

def novo_modelo():
    return RandomForestRegressor(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=SEMENTE)

**Validação separada por motor:** os 100 motores são divididos em 5 grupos. Cada
grupo é previsto por um modelo treinado só nos outros 4 — o modelo nunca viu o
motor que está prevendo.

In [5]:
previsao = np.zeros(len(treino))
for idx_treino, idx_prev in GroupKFold(n_splits=5).split(treino, groups=treino.motor):
    m = novo_modelo().fit(treino.iloc[idx_treino][ENTRADAS], treino.iloc[idx_treino]["alvo"])
    previsao[idx_prev] = m.predict(treino.iloc[idx_prev][ENTRADAS])
treino["previsto"] = previsao

## 4. Conferência com o teste oficial da NASA

Os 100 motores de teste param em algum ponto antes da falha, e a NASA informa
quantos ciclos ainda faltavam. É a comparação que a literatura usa, pela métrica
RMSE (erro típico, em ciclos).

In [6]:
modelo = novo_modelo().fit(treino[ENTRADAS], treino["alvo"])
ultimo = teste.groupby("motor").tail(1).merge(rul_teste, on="motor")
ultimo["previsto"] = modelo.predict(ultimo[ENTRADAS])
rmse = np.sqrt(((ultimo.previsto - ultimo.rul_real.clip(upper=TETO)) ** 2).mean())
print(f"RMSE no teste oficial FD001: {rmse:.1f} ciclos")

RMSE no teste oficial FD001: 16.4 ciclos


## 5. As duas políticas

**Troca por tempo fixo:** todo motor sai de operação no ciclo T. Falha quem durar
menos que T.

**Troca preditiva:** o alarme dispara quando a vida restante prevista cai abaixo de
uma margem M. A intervenção acontece **20 ciclos depois do alarme** — o prazo para
programar a parada, separar peça e mobilizar equipe. Falha quem não chegar até a
intervenção.

O prazo de 20 ciclos foi fixado antes de ver os resultados. Sem ele, a comparação
favoreceria a preditiva de forma injusta: ninguém para um equipamento no instante
do alerta.

**Desperdício** = ciclos de vida útil que o motor ainda tinha quando saiu de operação.

In [7]:
PRAZO_ACAO = 20
MARGENS = range(0, TETO + 1)

primeiro_alarme = {
    mo: {M: (d.loc[d.previsto <= M, "ciclo"].min() if (d.previsto <= M).any() else np.inf) for M in MARGENS}
    for mo, d in treino.groupby("motor")
}

def preditiva(M, motores):
    falhas, desperdicio = 0, []
    for mo in motores:
        intervencao = primeiro_alarme[mo][M] + PRAZO_ACAO
        if intervencao >= vida[mo]:
            falhas += 1
        else:
            desperdicio.append(vida[mo] - intervencao)
    return falhas, (np.mean(desperdicio) if desperdicio else np.nan)

def por_tempo(T, motores):
    v = vida[list(motores)]
    return int((v <= T).sum()), (v[v > T] - T).mean()

def calibrar(motores, max_falhas):
    M = next(M for M in MARGENS if preditiva(M, motores)[0] <= max_falhas)
    T = max(T for T in range(1, int(vida.max()) + 1) if por_tempo(T, motores)[0] <= max_falhas)
    return M, T

## 6. Comparação com o mesmo risco

Para cada nível de risco aceito, cada política recebe o seu melhor ajuste. As duas
são calibradas do mesmo jeito, nos mesmos 100 motores.

In [8]:
motores = list(vida.index)
linhas = []
for max_falhas in [0, 2, 5]:
    M, T = calibrar(motores, max_falhas)
    fp, dp = preditiva(M, motores)
    ft, dt = por_tempo(T, motores)
    linhas.append({"Risco aceito (falhas em 100)": max_falhas,
                   "Tempo fixo: troca no ciclo": T, "Tempo fixo: falhas": ft, "Tempo fixo: desperdício (ciclos)": round(dt, 1),
                   "Preditiva: margem": M, "Preditiva: falhas": fp, "Preditiva: desperdício (ciclos)": round(dp, 1)})
comparacao = pd.DataFrame(linhas).set_index("Risco aceito (falhas em 100)")
comparacao.T

Risco aceito (falhas em 100),0,2,5
Tempo fixo: troca no ciclo,127.0,136.0,146.0
Tempo fixo: falhas,0.0,2.0,4.0
Tempo fixo: desperdício (ciclos),79.3,71.8,63.3
Preditiva: margem,45.0,40.0,38.0
Preditiva: falhas,0.0,1.0,5.0
Preditiva: desperdício (ciclos),23.4,18.7,16.9


## 7. Teste de robustez

Calibrar e medir nos mesmos motores é otimista para as duas políticas. Então: 200
vezes, a regra de **zero falhas** é escolhida em 50 motores sorteados e aplicada nos
outros 50, que ela nunca viu.

In [9]:
rng = np.random.default_rng(SEMENTE)
res_p, res_t = [], []
for _ in range(200):
    embaralhado = rng.permutation(motores)
    A, B = embaralhado[:50], embaralhado[50:]
    M, T = calibrar(A, 0)
    res_p.append(preditiva(M, B))
    res_t.append(por_tempo(T, B))
res_p, res_t = np.array(res_p, dtype=float), np.array(res_t, dtype=float)
robustez = pd.DataFrame({
    "Falhas em cada 100 motores novos": [res_t[:, 0].mean() * 2, res_p[:, 0].mean() * 2],
    "Desperdício médio (ciclos)": [np.nanmean(res_t[:, 1]), np.nanmean(res_p[:, 1])],
}, index=["Troca por tempo fixo", "Troca preditiva"]).round(1)
robustez

,Falhas em cada 100 motores novos,Desperdício médio (ciclos)
Troca por tempo fixo,1.6,77.0
Troca preditiva,1.2,21.6


## 8. O número

In [10]:
M0, T0 = calibrar(motores, 0)
_, desp_p = preditiva(M0, motores)
_, desp_t = por_tempo(T0, motores)
aproveitado_t = 1 - desp_t / vida.mean()
aproveitado_p = 1 - desp_p / vida.mean()
print("Com zero falhas nas duas políticas:")
print(f"  troca por tempo fixo (ciclo {T0}) desperdiça {desp_t:.0f} ciclos por motor — aproveita {aproveitado_t:.0%} da vida útil")
print(f"  troca preditiva, com {PRAZO_ACAO} ciclos de prazo, desperdiça {desp_p:.0f} — aproveita {aproveitado_p:.0%}")

Com zero falhas nas duas políticas:
  troca por tempo fixo (ciclo 127) desperdiça 79 ciclos por motor — aproveita 62% da vida útil
  troca preditiva, com 20 ciclos de prazo, desperdiça 23 — aproveita 89%


## 9. Limitações

- **Os motores são simulados.** O C-MAPSS vem de um modelo de motor turbofan da
  NASA, com um único modo de falha (degradação do compressor de alta pressão) e uma
  única condição de operação. Equipamento real tem vários modos de falha e ruído
  que a simulação não tem.
- **Calibração nos mesmos motores.** A tabela principal escolhe a regra e mede nos
  mesmos 100 motores. O teste de robustez (seção 7) mostra que, em motores novos,
  as duas políticas passam a ter cerca de 1 a 2 falhas em cada 100 — e a diferença
  de desperdício se mantém.
- **Prazo de ação fixo.** 20 ciclos foi uma escolha. Um prazo maior exige margem
  maior e aumenta o desperdício da preditiva.
- **Custo não está na conta.** O desperdício está em ciclos. Monitorar sensores tem
  custo, e parada programada não é de graça — a decisão real compara desperdício
  evitado com o custo de manter o monitoramento.
- **Vida restante limitada a 125 ciclos.** O modelo não distingue motores muito
  longe do fim; isso não afeta a decisão de parar, que acontece na reta final.

## Capa do projeto

In [11]:
from matplotlib import font_manager
fonte = "Segoe UI" if any("Segoe UI" in f.name for f in font_manager.fontManager.ttflist) else "DejaVu Sans"
plt.rcParams["font.family"] = fonte
TINTA, TINTA2, APOIO, ACAO, METRICA = "#0f172a", "#475569", "#64748b", "#1d4ed8", "#0e7490"

fig = plt.figure(figsize=(10.8, 13.5), dpi=100, facecolor="#f8fafc")
fig.text(0.07, 0.935, "PROJETO D2 · VIDA ÚTIL DE MOTORES", fontsize=15, color=APOIO, fontweight="bold")
fig.text(0.07, 0.905, "Trocar por tempo é seguro.\nE caro.", fontsize=40, color=TINTA, fontweight="bold",
         va="top", linespacing=1.12)
fig.text(0.07, 0.715, f"{aproveitado_t:.0%}", fontsize=86, color="#94a3b8", fontweight="bold")
fig.text(0.33, 0.73, "→", fontsize=50, color=APOIO)
fig.text(0.43, 0.715, f"{aproveitado_p:.0%}", fontsize=86, color=METRICA, fontweight="bold")
fig.text(0.07, 0.69, "da vida útil aproveitada, com zero falhas\nnas duas políticas",
         fontsize=19, color=TINTA2, va="top", linespacing=1.35)

fig.text(0.07, 0.618, "■", fontsize=16, color=ACAO)
fig.text(0.095, 0.619, "vida usada com a preditiva", fontsize=15, color=TINTA2)
fig.text(0.43, 0.618, "■", fontsize=16, color="#cbd5e1")
fig.text(0.455, 0.619, "vida que ainda sobrava", fontsize=15, color=TINTA2)
ax = fig.add_axes([0.07, 0.15, 0.86, 0.41], facecolor="#f8fafc")
ordem = vida.sort_values().index
for i, mo in enumerate(ordem):
    ax.plot([0, vida[mo]], [i, i], color="#cbd5e1", lw=2.2, solid_capstyle="butt")
    intervencao = primeiro_alarme[mo][M0] + PRAZO_ACAO
    ax.plot([0, intervencao], [i, i], color=ACAO, lw=2.2, solid_capstyle="butt")
from matplotlib.transforms import blended_transform_factory
ax.axvline(T0, color=TINTA, lw=1.6, ymax=1.04, clip_on=False)
# Rótulo acima da área das barras, para não cobrir nenhuma.
ax.text(T0 + 4, 1.035, f"troca por tempo: todo motor sai no ciclo {T0}", fontsize=15, color=TINTA,
        fontweight="bold", va="bottom", transform=blended_transform_factory(ax.transData, ax.transAxes))
ax.text(vida.max() - 4, 8, "preditiva: cada motor\nsai perto do próprio fim", fontsize=15, color=ACAO,
        ha="right", fontweight="bold", linespacing=1.3)
ax.set_xlim(0, vida.max() * 1.02); ax.set_ylim(-1, len(ordem))
ax.set_yticks([]); ax.tick_params(colors=APOIO, labelsize=13)
ax.set_xlabel("ciclos de operação · cada linha é um dos 100 motores", fontsize=15, color=APOIO)
for lado in ["top", "right", "left"]:
    ax.spines[lado].set_visible(False)
ax.spines["bottom"].set_color("#e2e8f0")

fig.text(0.07, 0.06, f"Preditiva com {PRAZO_ACAO} ciclos de prazo para programar a parada.\n"
         "NASA C-MAPSS FD001 · motores simulados.", fontsize=14, color=APOIO, linespacing=1.4)
fig.savefig("capa.png", dpi=100, facecolor=fig.get_facecolor())
plt.close(fig)
print("capa.png salva")

capa.png salva
